In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/model_clean.csv")
y = "market_value_in_eur"

train, temp = train_test_split(df, test_size=0.30, random_state=42)
val, test = train_test_split(temp, test_size=0.50, random_state=42)

print("Trening:", len(train), " Validering:", len(val), " Test:", len(test))

Trening: 3773  Validering: 809  Test: 809


In [2]:
def evaluer(navn, y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rel = np.abs(y_pred - y_true) / y_true
    return {
        "modell": navn,
        "median_rel_feil": np.median(rel),
        "MAE_mill_eur": np.mean(np.abs(y_pred - y_true)) / 1e6,
    }

resultater = []

In [3]:
tabell = train.groupby(["league", "position"])[y].median().to_dict()
liga_median = train.groupby("league")[y].median().to_dict()
total_median = train[y].median()

def baseline_predict(data):
    return [
        tabell.get((l, p), liga_median.get(l, total_median))
        for l, p in zip(data["league"], data["position"])
    ]

pd.Series(tabell).unstack().round(-3)

,Attack,Defender,Goalkeeper,Midfield
BE1,2500000.0,2000000.0,900000.0,2350000.0
DK1,1200000.0,1200000.0,675000.0,800000.0
ES1,5000000.0,4000000.0,2900000.0,5000000.0
FR1,7000000.0,6000000.0,2500000.0,8000000.0
GB1,28000000.0,18000000.0,10000000.0,25000000.0
GR1,700000.0,500000.0,700000.0,850000.0
IT1,9000000.0,6000000.0,3250000.0,7000000.0
L1,8000000.0,5500000.0,4000000.0,8000000.0
NL1,1500000.0,1500000.0,2000000.0,1250000.0
PO1,1500000.0,1200000.0,1100000.0,2000000.0


In [4]:
pred_val = baseline_predict(val)
resultater.append(evaluer("Baseline (median liga × posisjon)", val[y], pred_val))

pd.DataFrame(resultater)

,modell,median_rel_feil,MAE_mill_eur
0,Baseline (median liga × posisjon),0.7,5.785878
